# Reinhardt Full-stack Demo — Manim scenes

Renders the concept-diagram scenes of the 15-minute full-stack demo
(see `source/conte.md`).

- Scene 1 — Opening / problem framing (0:00–1:30)
- Scene 2 — Reinhardt intro + polylithic architecture (1:30–2:30)
- Scene 9a — DI concept diagram (12:00–12:20)
- Scene 10 — Recap, comparison, CTA (13:30–15:00)

Use `-ql` for fast iteration and `-qh` for final render. Output MP4 files
land under `media/videos/manim/`.

Theme colors follow the conte memo: background `#1a1b26` (Tokyo Night),
Reinhardt accent `#f74c00`.

In [1]:
%load_ext manim

from manim import *

config.background_color = "#1a1b26"

BG        = "#1a1b26"
FG        = "#c0caf5"
ACCENT    = "#f74c00"   # Reinhardt brand orange
RUST      = "#ce422b"
DJANGO    = "#0c4b33"
MUTED     = "#565f89"
OK        = "#9ece6a"
WARN      = "#e0af68"

# Install on macOS if missing: `brew install --cask font-inter`
MONO_FONT = "Inter"
Text.set_default(font=MONO_FONT, color=FG)

The manim module is not an IPython extension.


## Scene 1 — Opening / problem framing (1:30)

In [2]:
%%manim -ql -v WARNING Scene1Opening

import numpy as np

class Scene1Opening(Scene):
    def construct(self):
        # 0:00–0:15  "Rust is fast." + speed bars
        headline = Text("Rust is fast.", font_size=72, weight=BOLD, color=FG)
        self.play(FadeIn(headline, shift=UP * 0.5), run_time=1.0)
        self.wait(0.5)
        self.play(headline.animate.to_edge(UP), run_time=0.6)

        langs  = ["Rust", "Go", "Python", "Ruby"]
        speeds = [1.00, 0.72, 0.18, 0.12]
        colors = [ACCENT, "#00ADD8", "#3572A5", "#701516"]
        BAR_LEFT = -3.0
        bars = VGroup()
        for i, (name, s, c) in enumerate(zip(langs, speeds, colors)):
            y = 1.5 - i * 1.0
            label = Text(name, font_size=28).move_to([-4.5, y, 0])
            bar = Rectangle(width=0.01, height=0.5, fill_opacity=1,
                            stroke_width=0, fill_color=c)
            bar.move_to([BAR_LEFT, y, 0], aligned_edge=LEFT)
            target = bar.copy().stretch_to_fit_width(8 * s)
            target.move_to([BAR_LEFT, y, 0], aligned_edge=LEFT)
            bars.add(label, bar, target)
            self.play(FadeIn(label), Transform(bar, target), run_time=0.5)
        self.wait(1.5)
        self.play(FadeOut(*self.mobjects))

        # 0:15–0:40  crate tile rain
        subtitle = Text("But building a web app takes...", font_size=44, color=MUTED)
        self.play(Write(subtitle), run_time=1.0)
        self.wait(0.3)
        self.play(subtitle.animate.to_edge(UP))

        # Row 0 (4 tiles) and row 1 (3 tiles), each row independently centered.
        ROW_Y = [1.0, -1.2]
        positions = {
            "axum":     (-4.5, ROW_Y[0]),
            "tower":    (-1.5, ROW_Y[0]),
            "sea-orm":  ( 1.5, ROW_Y[0]),
            "utoipa":   ( 4.5, ROW_Y[0]),
            "jwt":      (-3.0, ROW_Y[1]),
            "tracing":  ( 0.0, ROW_Y[1]),
            "sqlx-cli": ( 3.0, ROW_Y[1]),
        }
        tiles = {}
        tile_group = VGroup()
        for name, (x, y) in positions.items():
            t = Text(name, font_size=26, color=FG)
            box = SurroundingRectangle(t, color=MUTED, buff=0.15,
                                       stroke_width=2,
                                       fill_color=BG, fill_opacity=1.0)
            tile = VGroup(box, t).move_to([x, y, 0]).shift(UP * 6)
            tiles[name] = tile
            tile_group.add(tile)
        self.play(
            *[t.animate.shift(DOWN * 6) for t in tile_group],
            run_time=2.0, lag_ratio=0.05,
        )

        MARGIN = 0.2

        def h_arrow(src, dst):
            s = src.get_right() + RIGHT * MARGIN
            e = dst.get_left()  + LEFT  * MARGIN
            return Arrow(s, e, color=WARN, stroke_width=3,
                         buff=0.0, tip_length=0.2)

        def elbow_down_arrow(src, dst):
            s = src.get_bottom() + DOWN * MARGIN
            e = dst.get_top()    + UP   * MARGIN
            mid_y = (s[1] + e[1]) / 2
            shaft = VMobject(stroke_color=WARN, stroke_width=3)
            shaft.set_points_as_corners([
                s,
                np.array([s[0], mid_y, 0]),
                np.array([e[0], mid_y, 0]),
                e,
            ])
            tip = ArrowTriangleFilledTip(color=WARN, length=0.2).scale(1.0)
            tip.rotate(-PI / 2 - tip.tip_angle)
            tip.move_to(e)
            return VGroup(shaft, tip)

        horizontal_deps = [("axum", "tower"), ("tower", "sea-orm"), ("sea-orm", "utoipa")]
        vertical_deps   = [("tower", "jwt"), ("sea-orm", "tracing"), ("utoipa", "sqlx-cli")]

        arrows = VGroup()
        for s, d in horizontal_deps:
            arrows.add(h_arrow(tiles[s], tiles[d]))
        for s, d in vertical_deps:
            arrows.add(elbow_down_arrow(tiles[s], tiles[d]))

        self.play(LaggedStart(*[FadeIn(a) for a in arrows], lag_ratio=0.15))
        self.wait(1.5)

        # 0:40–1:10  Django side
        self.play(FadeOut(subtitle, tile_group, arrows))
        left  = Text("Rust", font_size=40, color=RUST).move_to(LEFT * 4 + UP * 2.5)
        right = Text("Django", font_size=40, color=OK).move_to(RIGHT * 4 + UP * 2.5)
        self.play(FadeIn(left), FadeIn(right))

        rust_items   = ["axum?", "sea-orm?", "utoipa?", "jwt?", "tower?"]
        django_items = ["ORM", "Admin", "Auth", "Migration", "Serializer"]
        for i, (r, d) in enumerate(zip(rust_items, django_items)):
            rt = Text(r, font_size=26, color=MUTED).move_to(LEFT * 4 + UP * (1.5 - i * 0.7))
            dt = Text(d, font_size=26, color=OK).move_to(RIGHT * 4 + UP * (1.5 - i * 0.7))
            self.play(FadeIn(rt), FadeIn(dt), run_time=0.35)
        self.wait(1.5)

        # 1:10–1:30  punch line
        self.play(FadeOut(*self.mobjects))
        punch = VGroup(
            Text("Django: 1 command.", font_size=48, color=OK),
            Text("Rust: 10+ crates.",  font_size=48, color=RUST),
            Text("?", font_size=140, color=ACCENT),
        ).arrange(DOWN, buff=0.4).move_to(ORIGIN)
        self.play(Write(punch[0]))
        self.play(Write(punch[1]))
        self.play(FadeIn(punch[2], scale=0.5))
        self.wait(1.5)

Manim Community v0.20.1

## Scene 2 — Reinhardt intro + polylithic architecture (1:00)

In [3]:
%%manim -ql -v WARNING Scene2Reinhardt

import numpy as np

class Scene2Reinhardt(Scene):
    def construct(self):
        # 1:30–1:45  logo + tagline
        logo = Text("Reinhardt", font_size=88, weight=BOLD, color=ACCENT)
        tag  = Text("🦀 Django's productivity, Rust's performance",
                    font_size=30, color=FG).next_to(logo, DOWN, buff=0.5)
        self.play(Write(logo), run_time=1.0)
        self.play(FadeIn(tag, shift=UP * 0.3), run_time=0.8)
        self.wait(1.0)
        self.play(logo.animate.scale(0.45).to_edge(UP),
                  FadeOut(tag))

        # 1:45–2:10  hub with nodes arranged as top row / bottom row.
        HUB_R = 0.8
        hub = Circle(radius=HUB_R, color=ACCENT, fill_opacity=1.0).set_fill(ACCENT)
        hub.set_z_index(2)
        hub_label = Text("reinhardt", font_size=22, color=BG).move_to(hub)
        hub_label.set_z_index(3)
        self.play(GrowFromCenter(hub), FadeIn(hub_label))

        crates = [
            "reinhardt-core",  "reinhardt-orm",    "reinhardt-di",      "reinhardt-auth",
            "reinhardt-admin", "reinhardt-api",    "reinhardt-graphql", "reinhardt-ws",
        ]
        TOP_Y, BOT_Y = 2.6, -2.6
        COLS = [-4.8, -1.6, 1.6, 4.8]
        layout = [
            (crates[0], COLS[0], TOP_Y),
            (crates[1], COLS[1], TOP_Y),
            (crates[2], COLS[2], TOP_Y),
            (crates[3], COLS[3], TOP_Y),
            (crates[4], COLS[0], BOT_Y),
            (crates[5], COLS[1], BOT_Y),
            (crates[6], COLS[2], BOT_Y),
            (crates[7], COLS[3], BOT_Y),
        ]
        MARGIN = 0.22
        nodes, edges = VGroup(), VGroup()
        for name, x, y in layout:
            t = Text(name, font_size=18, color=FG)
            box = SurroundingRectangle(t, color=MUTED, buff=0.12,
                                       stroke_width=1.5,
                                       fill_color=BG, fill_opacity=1.0)
            node = VGroup(box, t).move_to([x, y, 0])
            node.set_z_index(2)

            is_top = y > 0
            hub_exit = (hub.get_top() + UP * MARGIN) if is_top \
                       else (hub.get_bottom() + DOWN * MARGIN)
            node_entry = (node.get_bottom() + DOWN * MARGIN) if is_top \
                         else (node.get_top() + UP * MARGIN)
            corner = np.array([node.get_center()[0], hub_exit[1], 0])
            shaft = VMobject(stroke_color=MUTED, stroke_width=1.5)
            shaft.set_points_as_corners([hub_exit, corner, node_entry])
            shaft.set_z_index(1)

            nodes.add(node); edges.add(shaft)

        self.play(LaggedStart(*[Create(e) for e in edges], lag_ratio=0.08),
                  LaggedStart(*[FadeIn(n) for n in nodes], lag_ratio=0.08),
                  run_time=2.5)

        for n in nodes:
            self.play(n[0].animate.set_color(ACCENT), run_time=0.15)
            self.play(n[0].animate.set_color(MUTED),  run_time=0.15)
        self.wait(0.5)

        # 2:10–2:30  goal strip
        self.play(FadeOut(hub, hub_label, nodes, edges))
        goal = Text("Build a full-stack app in 15 minutes",
                    font_size=40, color=ACCENT).shift(UP * 0.8)
        steps = ["Model", "Migration", "API", "WASM UI", "Auth"]
        strip = VGroup(*[Text(s, font_size=28, color=FG) for s in steps])
        strip.arrange(RIGHT, buff=1.0).next_to(goal, DOWN, buff=0.8)
        arrows_between = VGroup(*[
            Arrow(strip[i].get_right(), strip[i + 1].get_left(),
                  color=MUTED, stroke_width=2, buff=0.1)
            for i in range(len(strip) - 1)
        ])
        self.play(FadeIn(goal))
        self.play(LaggedStart(*[FadeIn(t) for t in strip], lag_ratio=0.15),
                  LaggedStart(*[GrowArrow(a) for a in arrows_between], lag_ratio=0.15))
        self.wait(2.0)

Manim Community v0.20.1

## Scene 9a — DI concept diagram (0:20)

Visualizes how `#[inject]` parameters on a `#[server_fn]` are resolved at
compile time by walking the DI Container and its Provider Registry.

In [ ]:
%%manim -ql -v WARNING Scene9DI

import numpy as np

class Scene9DI(Scene):
    def construct(self):
        MARGIN = 0.35   # gap between outer box edge and arrow endpoint
        GAP    = 1.4    # gap between outer box edges

        # ── Signature ────────────────────────────────────────────────────────
        sig_lines = [
            "#[server_fn(pre_validate = true)]",
            "pub async fn publish_post(",
            "    input: PublishInput,",
            "    #[inject] AuthUser(user): AuthUser<User>,",
            "    #[inject] _: Guard<IsActiveUser>,",
            "    #[inject] posts: PostRepository,",
            ") -> Result<PostSerializer> { ... }",
        ]
        sig = VGroup(*[
            Text(line, font_size=14, color=FG, font=MONO_FONT)
            for line in sig_lines
        ]).arrange(DOWN, aligned_edge=LEFT, buff=0.42)
        sig_box = SurroundingRectangle(sig, color=FG, buff=0.22,
                                       stroke_width=2.0,
                                       fill_color=BG, fill_opacity=1.0)
        sig_grp = VGroup(sig_box, sig)

        # ── Slots / Providers ─────────────────────────────────────────────────
        def boxed(label, color, font_size=14):
            t = Text(label, font_size=font_size, color=FG, font=MONO_FONT)
            box = SurroundingRectangle(t, color=color, buff=0.14,
                                       stroke_width=1.5,
                                       fill_color=BG, fill_opacity=1.0)
            return VGroup(box, t)

        slot_auth  = boxed("AuthUser<User>",      ACCENT)
        slot_guard = boxed("Guard<IsActiveUser>", ACCENT)
        slot_repo  = boxed("PostRepository",      ACCENT)
        slot_w = max(slot_auth.width, slot_guard.width, slot_repo.width)
        for s in (slot_auth, slot_guard, slot_repo):
            s[0].stretch_to_fit_width(slot_w); s[0].move_to(s[1])

        p_token     = boxed("TokenAuthConfig",                         MUTED, font_size=13)
        p_is_active = boxed("IsActiveUser",                            MUTED, font_size=13)
        p_factory   = boxed("#[injectable_factory]\n post_repository", MUTED, font_size=13)
        prov_w = max(p_token.width, p_is_active.width, p_factory.width)
        for p in (p_token, p_is_active, p_factory):
            p[0].stretch_to_fit_width(prov_w); p[0].move_to(p[1])

        # ── Measure box widths via temp groups ────────────────────────────────
        _c = VGroup(Text("DI Container", font_size=20, weight=BOLD),
                    slot_auth.copy(), slot_guard.copy(), slot_repo.copy()).arrange(DOWN, buff=0.35)
        cont_box_w = SurroundingRectangle(_c, buff=0.25).width

        _r = VGroup(Text("Provider Registry", font_size=20, weight=BOLD),
                    p_token.copy(), p_is_active.copy(), p_factory.copy()).arrange(DOWN, buff=0.35)
        reg_box_w  = SurroundingRectangle(_r, buff=0.25).width

        # ── Compute x-positions so the 3-box group is centered ────────────────
        sig_w   = sig_grp.width
        total_w = sig_w + GAP + cont_box_w + GAP + reg_box_w
        left_x  = -total_w / 2

        sig_cx   = left_x + sig_w / 2
        cont_lx  = left_x + sig_w + GAP
        cont_cx  = cont_lx + cont_box_w / 2
        reg_lx   = cont_lx + cont_box_w + GAP
        reg_cx   = reg_lx + reg_box_w / 2

        sig_grp.move_to([sig_cx, 0, 0])

        # y-coords of the three #[inject] lines
        y_auth  = sig[3].get_center()[1]
        y_guard = sig[4].get_center()[1]
        y_repo  = sig[5].get_center()[1]

        # ── Place DI Container ────────────────────────────────────────────────
        slot_auth.move_to([cont_cx,  y_auth,  0])
        slot_guard.move_to([cont_cx, y_guard, 0])
        slot_repo.move_to([cont_cx,  y_repo,  0])
        container_title = Text("DI Container", font_size=20,
                               color=ACCENT, weight=BOLD)
        container_title.move_to([cont_cx, y_auth + 0.85, 0])
        container_box = SurroundingRectangle(
            VGroup(container_title, slot_auth, slot_guard, slot_repo),
            color=ACCENT, buff=0.25, stroke_width=2,
            fill_color=BG, fill_opacity=1.0,
        )

        # ── Place Provider Registry ───────────────────────────────────────────
        p_token.move_to([reg_cx,     y_auth,  0])
        p_is_active.move_to([reg_cx, y_guard, 0])
        p_factory.move_to([reg_cx,   y_repo,  0])
        reg_title = Text("Provider Registry", font_size=20,
                         color=MUTED, weight=BOLD)
        reg_title.move_to([reg_cx, y_auth + 0.85, 0])
        reg_box = SurroundingRectangle(
            VGroup(reg_title, p_token, p_is_active, p_factory),
            color=MUTED, buff=0.25, stroke_width=2,
            fill_color=BG, fill_opacity=1.0,
        )

        # ── Arrow helper (uses actual outer-box edges) ────────────────────────
        def h_arrow(x_start, x_end, y, color):
            s = np.array([x_start + MARGIN, y, 0])
            e = np.array([x_end   - MARGIN, y, 0])
            return Arrow(s, e, color=color, stroke_width=3,
                         buff=0.0, tip_length=0.22,
                         max_tip_length_to_length_ratio=0.3)

        sig_right  = sig_box.get_right()[0]
        cont_left  = container_box.get_left()[0]
        cont_right = container_box.get_right()[0]
        reg_left   = reg_box.get_left()[0]

        # ── Animate ───────────────────────────────────────────────────────────
        self.play(FadeIn(sig_box), Write(sig), run_time=1.2)
        self.play(sig[3].animate.set_color(ACCENT),
                  sig[4].animate.set_color(ACCENT),
                  sig[5].animate.set_color(ACCENT), run_time=0.8)
        self.wait(0.4)

        self.play(FadeIn(container_box), FadeIn(container_title),
                  FadeIn(slot_auth), FadeIn(slot_guard), FadeIn(slot_repo),
                  run_time=0.8)

        a1 = h_arrow(sig_right, cont_left, y_auth,  ACCENT)
        a2 = h_arrow(sig_right, cont_left, y_guard, ACCENT)
        a3 = h_arrow(sig_right, cont_left, y_repo,  ACCENT)
        self.play(FadeIn(a1), FadeIn(a2), FadeIn(a3), run_time=0.8)

        resolved = Text("resolved at compile time", font_size=22,
                        color=ACCENT).to_edge(DOWN, buff=0.6)
        self.play(Write(resolved))
        self.wait(0.8)

        self.play(FadeIn(reg_box), FadeIn(reg_title),
                  FadeIn(p_token), FadeIn(p_is_active), FadeIn(p_factory),
                  run_time=0.8)

        r1 = h_arrow(cont_right, reg_left, y_auth,  MUTED)
        r2 = h_arrow(cont_right, reg_left, y_guard, MUTED)
        r3 = h_arrow(cont_right, reg_left, y_repo,  MUTED)
        self.play(FadeIn(r1), FadeIn(r2), FadeIn(r3), run_time=0.8)
        self.wait(2.0)

## Scene 10 — Recap, comparison, CTA (1:30)

In [5]:
%%manim -ql -v WARNING Scene10Closing

class Scene10Closing(Scene):
    def construct(self):
        # 13:30–14:10  checklist + line counter
        checklist = VGroup(*[
            Text(f"✅ {label}", font_size=32, color=OK)
            for label in ["Model", "Migration", "CRUD API", "WASM Frontend", "Admin", "Auth + DI"]
        ]).arrange(DOWN, aligned_edge=LEFT, buff=0.3).to_edge(LEFT, buff=1.2)

        counter = Text("≈ 80 lines of Rust",
                       font_size=46, color=ACCENT).to_edge(RIGHT, buff=1.2)

        self.play(LaggedStart(*[FadeIn(item, shift=RIGHT * 0.3) for item in checklist],
                              lag_ratio=0.2))
        self.play(Write(counter))
        self.wait(2.5)

        # 14:10–14:50  CTA
        self.play(FadeOut(checklist, counter))
        cmd = Text("$ cargo install reinhardt-admin-cli",
                   font_size=36, color=FG, font=MONO_FONT)
        repo = Text("github.com/kent8192/reinhardt-web",
                    font_size=28, color=MUTED).next_to(cmd, DOWN, buff=0.5)
        site = Text("reinhardt-web.dev",
                    font_size=28, color=ACCENT).next_to(repo, DOWN, buff=0.3)
        self.play(Write(cmd))
        self.play(FadeIn(repo), FadeIn(site))
        self.wait(2.0)

        # 14:50–15:00  closing tagline
        self.play(FadeOut(cmd, repo, site))
        closer = Text("Django's productivity. Rust's performance.",
                      font_size=40, weight=BOLD, color=ACCENT)
        self.play(FadeIn(closer, scale=0.9))
        self.wait(2.0)
        self.play(FadeOut(closer))

Manim Community v0.20.1